In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import os
import mne
import matplotlib.pyplot as plt
import gc
import traceback

In [ ]:
import os
import mne

# Root EEG directory
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# Get numeric subject directories
subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    print("\n" + "=" * 50)
    print(f"🧠 Processing Subject {subj}")
    print("=" * 50 + "\n")

    subj_path = os.path.join(root_dir, subj)

    # Find cleaned ICA files
    session_files = sorted([f for f in os.listdir(subj_path) if f.endswith('_clean_after_ica.fif')])

    if not session_files:
        print(f"⚠️ No ICA-cleaned FIF files found for {subj}. Skipping.")
        continue

    print(f"🔍 Found ICA-cleaned files: {session_files}")

    for session_file in session_files:
        reref_file = session_file.replace('_clean_after_ica.fif', '_ica_reref_raw.fif')
        reref_path = os.path.join(subj_path, reref_file)

        # Check if already processed
        if os.path.exists(reref_path):
            print(f"✅ Already re-referenced: {reref_file} — skipping.")
            continue

        session_path = os.path.join(subj_path, session_file)
        print(f"📂 Loading: {session_path}")

        try:
            raw = mne.io.read_raw_fif(session_path, preload=True)

            # Check for mastoid channels
            if not all(ch in raw.ch_names for ch in ['E57', 'E100']):
                print(f"⚠️ Mastoid channels missing in {session_file}. Skipping.")
                continue

            # Apply linked mastoid reference
            lm = raw.copy().pick_channels(['E57']).get_data()
            rm = raw.copy().pick_channels(['E100']).get_data()
            mastoid_avg = (lm + rm) / 2
            raw._data = raw.get_data() - mastoid_avg

            print(f"✅ Re-referenced {session_file} using linked mastoids (E57/E100)")

            # Save re-referenced file
            raw.save(reref_path, overwrite=True)
            print(f"💾 Saved re-referenced file: {reref_path}")

        except Exception as e:
            print(f"❌ Error processing {session_file}: {e}")

